#  RT-DETR Pothole Detection
**Real-Time DEtection TRansformer** fine-tuned for pothole detection



In [1]:
# Installs ultralytics, kagglehub, seaborn and tqdm.
import subprocess, sys


def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])


pip_install("ultralytics", "kagglehub", "seaborn", "tqdm")
print("Dependencies installed")

Dependencies installed



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT, SAVE_DIR and OUTPUT_DIR accordingly.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_JUPYTER = not ON_KAGGLE and not ON_COLAB

if ON_KAGGLE:
    print("Running on KAGGLE")
    ROOT = "/kaggle/working"
    SAVE_DIR = "/kaggle/working/saved_models"
elif ON_COLAB:
    print("Running on GOOGLE COLAB")
    ROOT = "/content"
    SAVE_DIR = "/content/saved_models"
else:
    print("Running on LOCAL JUPYTER")
    ROOT = "."
    SAVE_DIR = "./saved_models"

OUTPUT_DIR = os.path.join(ROOT, "rtdetr_results")
DATA_DIR = os.path.join(ROOT, "data")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"ROOT       : {ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"SAVE_DIR   : {SAVE_DIR}")

Running on LOCAL JUPYTER
ROOT       : .
OUTPUT_DIR : ./rtdetr_results
SAVE_DIR   : ./saved_models


In [3]:
# Loads all shared imports and confirms GPU availability.
import torch, numpy as np, pandas as pd, cv2
import time, json, glob, warnings, shutil, yaml
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device      : {DEVICE}")
print(f"Torch       : {torch.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"OpenCV      : {cv2.__version__}")

Device      : cuda
Torch       : 2.10.0+cu128
NumPy       : 2.2.6
OpenCV      : 4.13.0


In [4]:
# Defines the fixed evaluation config, CLASS_NAMES, CONF_THRESH, IOU_THRESH, IMG_SIZE, and the SKIP_TRAINING switch.
CLASS_NAMES = ["pothole"]
CONF_THRESH = 0.25
IOU_THRESH = 0.45
IMG_SIZE = 640
MAX_IMAGES = None
SKIP_TRAINING = True  # True = load saved weights, False = train

PALETTE = {
    "YOLOv8m": "#00d4ff",
    "YOLOv10m": "#3b82f6",
    "YOLOv11m": "#6366f1",
    "Faster R-CNN": "#f97316",
    "SSD-VGG16": "#ec4899",
    "YOLO+FRCNN Ensemble": "#a855f7",
    "YOLOv8m+CBAM": "#22c55e",
    "YOLOv8m+CoordAttn": "#eab308",
    "RT-DETR": "#10b981",
}

print("Config loaded")
print(f"   CONF_THRESH  = {CONF_THRESH}")
print(f"   IOU_THRESH   = {IOU_THRESH}")
print(f"   IMG_SIZE     = {IMG_SIZE}")
print(f"   MAX_IMAGES   = {MAX_IMAGES}")
print(f"   SKIP_TRAINING= {SKIP_TRAINING}")

Config loaded
   CONF_THRESH  = 0.25
   IOU_THRESH   = 0.45
   IMG_SIZE     = 640
   MAX_IMAGES   = None
   SKIP_TRAINING= True


 Dataset Loading

In [5]:
# Downloads all three Kaggle pothole datasets through kagglehub.
import kagglehub

print("Downloading datasets via kagglehub ...")
path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")

DATASET_ROOTS = {
    "chitholian": path_1,
    "andrewmvd": path_2,
    "ashishkumar": path_3,
}
print(f"Dataset roots: {DATASET_ROOTS}")

Dataset roots: {'chitholian': '/home/vr3/.cache/kagglehub/datasets/chitholian/annotated-potholes-dataset/versions/1', 'andrewmvd': '/home/vr3/.cache/kagglehub/datasets/andrewmvd/pothole-detection/versions/1', 'ashishkumar': '/home/vr3/.cache/kagglehub/datasets/ashishkumarak/training-setzip/versions/1'}


In [6]:
# Defines the unified recursive XML annotation loader and loads the chitholian dataset.
def load_annotated_potholes(root, max_imgs=None):
    root = Path(root)
    records = []
    for img_path in list(root.rglob("*.jpg")) + list(root.rglob("*.png")):
        xml_path = img_path.with_suffix(".xml")
        if not xml_path.exists():
            xml_path = img_path.parent.parent / "annotations" / (img_path.stem + ".xml")
        gt_boxes = []
        if xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                for obj in tree.findall("object"):
                    bb = obj.find("bndbox")
                    gt_boxes.append(
                        {
                            "label": (obj.find("name").text or "pothole").lower(),
                            "xmin": float(bb.find("xmin").text),
                            "ymin": float(bb.find("ymin").text),
                            "xmax": float(bb.find("xmax").text),
                            "ymax": float(bb.find("ymax").text),
                        }
                    )
            except Exception:
                pass
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
        if max_imgs is not None and len(records) >= max_imgs:
            break
    print(f"[Annotated Potholes] images={len(records)}")
    return records


records_1 = load_annotated_potholes(DATASET_ROOTS["chitholian"])


[Annotated Potholes] images=665


In [7]:
# Loads the andrewmvd dataset through the same unified loader.
records_2 = load_annotated_potholes(DATASET_ROOTS["andrewmvd"])


[Annotated Potholes] images=665


In [8]:
# Loads the ashishkumarak dataset through the CSV loader, since this source uses train/labels.csv rather than XML.
import pandas as pd


def load_ashishkumar_csv(root, max_imgs=None):
    root = Path(root)
    csv_path = root / "train" / "labels.csv"
    img_dir = root / "train" / "images"
    df = pd.read_csv(csv_path)
    grouped = df.groupby("ImageID")

    records = []
    img_paths = sorted(img_dir.glob("*.jpg"))[:max_imgs]
    for img_path in img_paths:
        gt_boxes = []
        if img_path.name in grouped.groups:
            for _, row in grouped.get_group(img_path.name).iterrows():
                gt_boxes.append(
                    {
                        "label": "pothole",
                        "xmin": float(row["XMin"]),
                        "ymin": float(row["YMin"]),
                        "xmax": float(row["XMax"]),
                        "ymax": float(row["YMax"]),
                    }
                )
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    print(
        f"[ashishkumar CSV] images={len(records)} ({sum(len(r['gt_boxes']) for r in records)} gt boxes)"
    )
    return records


records_3 = load_ashishkumar_csv(DATASET_ROOTS["ashishkumar"])


[ashishkumar CSV] images=674 (1371 gt boxes)


In [9]:
# Tags each record with its source, merges all three, removes duplicates by MD5 and then by normalized-pixel comparison, and defines the canonical train/val split.
for r in records_1:
    r["source"] = "annotated_dataset"
for r in records_2:
    r["source"] = "voc_dataset"
for r in records_3:
    r["source"] = "csv_dataset"

records = records_1 + records_2 + records_3
print(f"Total images before dedup: {len(records)}")


import numpy as np
from PIL import Image

NORM_SIZE = (64, 64)
DEDUP_THRESHOLD = 1.0  # mean abs pixel diff (0-255 scale), true duplicates
# measured at 0.10-0.50, unrelated images much higher


def normalized_pixels(path):
    with Image.open(path) as img:
        return np.asarray(
            img.convert("L").resize(NORM_SIZE, Image.LANCZOS), dtype=np.float32
        ).ravel()


annotated_only = [r for r in records if r["gt_boxes"]]
print("Computing normalized pixel arrays for dedup ...")
all_arrs = np.stack([normalized_pixels(r["image_path"]) for r in annotated_only])

keep_mask = np.ones(len(annotated_only), dtype=bool)
seen_arrs = []
for i in range(len(annotated_only)):
    if not keep_mask[i]:
        continue
    if seen_arrs:
        diffs = np.abs(np.stack(seen_arrs) - all_arrs[i]).mean(axis=1)
        if diffs.min() < DEDUP_THRESHOLD:
            keep_mask[i] = False
            continue
    seen_arrs.append(all_arrs[i])

n_before = len(records)
records = [r for r, keep in zip(annotated_only, keep_mask) if keep]
n_after = len(records)
print(f"Total images after dedup: {n_after}")
print(f"Duplicates removed: {n_before - n_after}")


import random as _random

_random.seed(42)
_annotated = [r for r in records if r["gt_boxes"]]
_shuffled = _annotated.copy()
_random.shuffle(_shuffled)
_split_idx = int(len(_shuffled) * 0.8)
train_recs = _shuffled[:_split_idx]
val_recs = _shuffled[_split_idx:]
print(
    f"\nCanonical split: train={len(train_recs)}  val={len(val_recs)}  (seed=42, shuffled)"
)

# Quick dataset summary
src_counts = defaultdict(int)
for r in records:
    src_counts[r["source"]] += 1
for src, cnt in src_counts.items():
    print(f"  {src}: {cnt} images")

Total images before dedup: 2004
Computing normalized pixel arrays for dedup ...
Total images after dedup: 926
Duplicates removed: 1078

Canonical split: train=740  val=186  (seed=42, shuffled)
  annotated_dataset: 665 images
  voc_dataset: 9 images
  csv_dataset: 252 images


In [10]:
# Reports the dataset composition after deduplication, images, boxes and per-source breakdown, as given in the paper's dataset table.

total_boxes = sum(len(r["gt_boxes"]) for r in records)
print(f"Final dataset: {len(records)} images, {total_boxes} ground-truth boxes")
print(f"Mean boxes per image: {total_boxes / len(records):.4f}")


from collections import defaultdict

per_source_imgs = defaultdict(int)
per_source_boxes = defaultdict(int)
for r in records:
    src = r.get("source", "unknown")
    per_source_imgs[src] += 1
    per_source_boxes[src] += len(r["gt_boxes"])
for src in per_source_imgs:
    print(f"  {src}: {per_source_imgs[src]} images, {per_source_boxes[src]} boxes")

Final dataset: 926 images, 2354 ground-truth boxes
Mean boxes per image: 2.5421
  annotated_dataset: 665 images, 1740 boxes
  voc_dataset: 9 images, 31 boxes
  csv_dataset: 252 images, 583 boxes
